# Themes v0 — minimal

Goal: a **ranked table** of news themes labelled `aligned` / `novel` against ICB Sectors.

Pipeline:

1. ICB Subsector definitions → embedding → **Sector centroids** (45)
2. Sample one week of headlines → strip wire prefixes & dates → **FinLang embeddings**
3. **BERTopic** on the embeddings
4. For each theme: cosine to every Sector centroid → **nearest sector + max cosine + bucket**
5. **Drill-down HTML** with top headlines per theme

Out of scope (deferred to v1+): rolling windows / persistence, ticker tagging, news-volume slope, 2D map.

In [1]:
from __future__ import annotations

import re
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import polars as pl
import torch
import umap
from bertopic import BERTopic
from hdbscan import HDBSCAN
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
NEWS_PATH = PROJECT_ROOT / "data" / "raw" / "raw_news_2025.csv"
ICB_PATH = PROJECT_ROOT / "notebooks" / "input" / "icb-structure-and-definitions.xlsx"
OUTPUT_HTML = PROJECT_ROOT / "notebooks" / "output" / "themes_v0.html"
OUTPUT_HTML.parent.mkdir(parents=True, exist_ok=True)

EMBEDDING_MODEL = "FinLang/finance-embeddings-investopedia"
RANDOM_SEED = 42

DATE_START = "2025-01-01T00:00:00+00:00"
DATE_END = "2025-01-08T00:00:00+00:00"  # 1 week
SAMPLE_N = 10_000                        # cap for fast iteration

MIN_TOPIC_SIZE = 60
MIN_SAMPLES = 10
TAU_COV = 0.30                           # cos >= TAU_COV → "aligned"

DEVICE = (
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)
print(f"Device: {DEVICE}")

Device: mps


## 1. ICB Sector centroids

Embed each Subsector (`name + definition`) with FinLang, then average per ICB Sector → 45 normalized vectors.

In [2]:
embedder = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE)

icb_raw = pd.read_excel(ICB_PATH, sheet_name="Mappable", header=1).dropna(subset=["Subsector"])
icb = pd.DataFrame(
    {
        "industry": icb_raw["Industry"].str.strip(),
        "sector": icb_raw["Sector"].str.strip(),
        "subsector": icb_raw["Subsector"].str.strip(),
        "definition": icb_raw["Definition"].fillna("").str.strip(),
    }
).reset_index(drop=True)

sub_texts = (icb["subsector"] + ". " + icb["definition"]).tolist()
sub_emb = embedder.encode(
    sub_texts, batch_size=32, show_progress_bar=True,
    convert_to_numpy=True, normalize_embeddings=True,
)

sector_meta = (
    icb.groupby("sector", as_index=False)
    .agg(industry=("industry", "first"), n_subsectors=("subsector", "count"))
    .sort_values(["industry", "sector"])  # group sectors by Industry for the Sankey
    .reset_index(drop=True)
)
sector_emb = np.stack(
    [sub_emb[(icb["sector"] == s).to_numpy()].mean(axis=0) for s in sector_meta["sector"]]
)
sector_emb /= np.linalg.norm(sector_emb, axis=1, keepdims=True)
print(f"Sectors: {len(sector_meta)}  (centroids of {len(icb)} subsectors)")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Sectors: 45  (centroids of 173 subsectors)


## 2. News sample + preprocessing

Load one week of headlines (dedupe by text) → uniform random sample → strip wire prefixes (`BSECorpAnn: …`) and dates.

In [3]:
DATE_PATTERNS = (
    r"\b\d{4}[/\-]\d{1,2}[/\-]\d{1,2}\b",
    r"\b\d{1,2}[/\-]\d{1,2}[/\-]\d{2,4}\b",
    r"\b(?:jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)[a-z]*\.?\s+\d{1,2},?\s+\d{4}\b",
    r"\bq[1-4]\s+\d{4}\b",
)


def strip_colon_prefix(text: str) -> str:
    """Drop short wire-source prefixes like 'BSECorpAnn:', 'Dawn News.pk:', 'Assam Sentinel:'.

    Only strips when the prefix is short (≤ 30 chars, ≤ 4 words) so we don't
    accidentally chop real content like 'S. Korea's FSS Refers Korea Zinc Execs to Prosecutors: Yonhap'.
    Applied twice to handle stacked prefixes like 'Reuters: NDTV: …'.
    """
    for _ in range(2):
        if not text or ":" not in text:
            return text
        prefix, _, rest = text.partition(":")
        prefix, rest = prefix.strip(), rest.strip()
        if not prefix or not prefix[0].isalpha():
            return text
        if len(prefix) > 30 or len(prefix.split()) > 4:
            return text
        if not rest:
            return text
        text = rest
    return text


def strip_dates(text: str) -> str:
    for pat in DATE_PATTERNS:
        text = re.sub(pat, " ", text, flags=re.IGNORECASE)
    text = re.sub(r":\s*\d+\b", ":", text)
    text = re.sub(r"\b\d+\b", " ", text)
    text = re.sub(r":\s*$", "", text)
    return re.sub(r"\s+", " ", text).strip()


def normalize_headline(text: str) -> str:
    return strip_dates(strip_colon_prefix(text))


corpus = (
    pl.scan_csv(NEWS_PATH)
    .select(["Headline", "CaptureTime"])
    .with_columns(pl.col("CaptureTime").str.to_datetime(time_zone="UTC"))
    .filter(
        (pl.col("CaptureTime") >= pl.lit(DATE_START).str.to_datetime(time_zone="UTC"))
        & (pl.col("CaptureTime") < pl.lit(DATE_END).str.to_datetime(time_zone="UTC"))
        & pl.col("Headline").is_not_null()
        & (pl.col("Headline").str.len_chars() > 0)
    )
    .unique(subset=["Headline"])
    .collect()
)
print(f"Unique headlines in window: {corpus.height:,}")

if corpus.height > SAMPLE_N:
    corpus = corpus.sample(n=SAMPLE_N, seed=RANDOM_SEED, shuffle=True)

raw_headlines = corpus["Headline"].to_list()
norm_headlines = [normalize_headline(h) for h in raw_headlines]
keep = [i for i, t in enumerate(norm_headlines) if t.strip()]
raw_headlines = [raw_headlines[i] for i in keep]
norm_headlines = [norm_headlines[i] for i in keep]
print(f"Headlines used:             {len(norm_headlines):,}\n")
print("Raw vs normalized:")
for r, n in zip(raw_headlines[:3], norm_headlines[:3]):
    print(f"  raw:  {r}")
    print(f"  norm: {n}\n")

Unique headlines in window: 151,122
Headlines used:             9,358

Raw vs normalized:
  raw:  NHL: Caufield, Dach lift Canadiens past Avalanche in shootout
  norm: Caufield, Dach lift Canadiens past Avalanche in shootout

  raw:  Xinhua: India's forex reserves drop over 4 bln USD
  norm: India's forex reserves drop over bln USD

  raw:  Indus & Pru Inv: Window closure notice 30.09.2024  - Annual Reports Annual Report : 2023-2024 Annual Report : 2022-2023 Annual
  norm: Window closure notice . . - Annual Reports Annual Report :- Annual Report :- Annual



## 3. Embed + BERTopic

In [4]:
news_emb = embedder.encode(
    norm_headlines, batch_size=64, show_progress_bar=True,
    convert_to_numpy=True, normalize_embeddings=True,
)
print(f"News embeddings: {news_emb.shape}")

bertopic_vectorizer = CountVectorizer(
    stop_words="english",
    token_pattern=r"(?u)\b[a-zA-Z]{3,}\b",
    min_df=2, max_df=0.9,
)
bertopic_umap = umap.UMAP(
    n_neighbors=15, n_components=5, min_dist=0.0,
    metric="cosine", random_state=RANDOM_SEED,
)
bertopic_hdbscan = HDBSCAN(
    min_cluster_size=MIN_TOPIC_SIZE, min_samples=MIN_SAMPLES,
    metric="euclidean", cluster_selection_method="eom", prediction_data=True,
)
topic_model = BERTopic(
    embedding_model=embedder,
    umap_model=bertopic_umap,
    hdbscan_model=bertopic_hdbscan,
    vectorizer_model=bertopic_vectorizer,
    verbose=False,
)
topics, _ = topic_model.fit_transform(norm_headlines, news_emb)
topics_arr = np.array(topics)

n_topics = len(set(topics)) - (1 if -1 in topics else 0)
print(f"BERTopic topics (excl. -1): {n_topics}")
print(f"Outliers:                   {(topics_arr == -1).mean():.1%}")

Batches:   0%|          | 0/147 [00:00<?, ?it/s]

News embeddings: (9358, 768)


BERTopic topics (excl. -1): 39
Outliers:                   31.8%


## 4. Ranked table — theme × sector

For each theme: normalized centroid of its headline embeddings, max cosine to the 45 Sector centroids, and bucket (`aligned` if `max_cos ≥ TAU_COV`, else `novel`). Sorted by `max_cos` descending — read top-to-bottom from "most sector-aligned" to "most novel".

In [5]:
def keywords(topic_id: int, n_words: int = 6) -> str:
    return ", ".join(w for w, _ in (topic_model.get_topic(topic_id) or [])[:n_words])


valid_topics = sorted(t for t in set(topics) if t != -1)
centroids = []
rows = []
for t in valid_topics:
    mask = topics_arr == t
    c = news_emb[mask].mean(axis=0)
    c /= np.linalg.norm(c) + 1e-12
    centroids.append(c)
    rows.append({"topic": t, "n_docs": int(mask.sum()), "keywords": keywords(t)})
centroids = np.stack(centroids)

sim = centroids @ sector_emb.T              # (n_topics, 45)
nearest = sim.argmax(axis=1)

themes = pd.DataFrame(rows)
themes["max_cos"] = sim.max(axis=1).round(3)
themes["nearest_sector"] = sector_meta["sector"].to_numpy()[nearest]
themes["nearest_industry"] = sector_meta["industry"].to_numpy()[nearest]
themes["bucket"] = np.where(themes["max_cos"] >= TAU_COV, "aligned", "novel")
themes = themes.sort_values("max_cos", ascending=False).reset_index(drop=True)

print(
    f"aligned: {(themes['bucket'] == 'aligned').sum()}   "
    f"novel: {(themes['bucket'] == 'novel').sum()}\n"
)
themes[["topic", "n_docs", "bucket", "nearest_sector", "max_cos", "keywords"]]

aligned: 22   novel: 17



,topic,n_docs,bucket,nearest_sector,max_cos,keywords
0,4,298,aligned,Life Insurance,0.612,"health, study, trends, risk, predictions, care"
1,18,128,aligned,Non-life Insurance,0.587,"csr, fwp, menu, open, london, city"
2,23,96,aligned,"Gas, Water and Multi-utilities",0.581,"energy, sector, africa, bills, electricity, ug..."
3,25,95,aligned,"Gas, Water and Multi-utilities",0.533,"submits, electric, hydro, accession, motion, n..."
4,22,97,aligned,Real Estate Investment Trusts,0.533,"fund, income, trust, llc, csr, funds"
5,24,95,aligned,Finance and Credit Services,0.526,"analysis, investment, term, advice, long, trade"
6,14,148,aligned,Automobiles and Parts,0.467,"sales, dec, tesla, ford, vehicles, revenue"
7,15,148,aligned,Media,0.460,"netflix, documentary, foundation, shows, watch..."
8,36,63,aligned,"Gas, Water and Multi-utilities",0.440,", , , , ,"
9,31,70,aligned,Real Estate Investment Trusts,0.426,"dividend, monthly, declares, return, ended, eq..."


## 5. Sankey + drill-down HTML

**Sankey** — left: themes (color = bucket), right: ICB Sectors **stacked by Industry** (color = Industry). Each link is drawn when `cosine ≥ TAU_SHOW`; width ∝ `cosine × n_docs`. Themes that match no sector flow into a gray `(no sector match)` sink.

**Drill-down** below the chart: for each theme, the 10 headlines most similar to the theme centroid.

In [6]:
import html as html_lib

TAU_SHOW = 0.25                # minimum cosine for a theme→sector link
TOP_HEADLINES_PER_THEME = 10

# --- Sankey nodes ---
INDUSTRY_PALETTE = px.colors.qualitative.Bold + px.colors.qualitative.Set3
INDUSTRY_ORDER = sector_meta["industry"].drop_duplicates().tolist()
INDUSTRY_COLORS = {ind: INDUSTRY_PALETTE[i % len(INDUSTRY_PALETTE)] for i, ind in enumerate(INDUSTRY_ORDER)}

theme_labels = []
for _, r in themes.iterrows():
    short_kws = ", ".join(k.strip() for k in (r["keywords"] or "").split(",")[:2] if k.strip())
    theme_labels.append(f"T{int(r['topic'])}: {short_kws[:32]}")
sector_labels = sector_meta["sector"].tolist()
novel_label = "(no sector match)"

node_labels = theme_labels + sector_labels + [novel_label]
n_themes, n_sectors = len(theme_labels), len(sector_labels)
novel_idx = n_themes + n_sectors

theme_colors = ["#1a7f37" if b == "aligned" else "#bf5700" for b in themes["bucket"]]
sector_colors = [INDUSTRY_COLORS[ind] for ind in sector_meta["industry"]]
node_colors = theme_colors + sector_colors + ["#999"]

# --- Sankey links ---
sources, targets, values, link_colors, link_hover = [], [], [], [], []
for i, row in themes.iterrows():
    t = int(row["topic"])
    cos_row = sim[valid_topics.index(t)]
    above = [(j, c) for j, c in enumerate(cos_row) if c >= TAU_SHOW]
    if above:
        link_rgba = "rgba(26,127,55,0.35)" if row["bucket"] == "aligned" else "rgba(191,87,0,0.35)"
        for j, c in above:
            sources.append(i)
            targets.append(n_themes + j)
            values.append(float(c * row["n_docs"]))
            link_colors.append(link_rgba)
            link_hover.append(
                f"theme {t}: {row['keywords']}<br>→ {sector_labels[j]} "
                f"({sector_meta['industry'].iloc[j]})<br>cos={c:.2f} · n_docs={row['n_docs']}"
            )
    else:
        sources.append(i)
        targets.append(novel_idx)
        values.append(float(row["n_docs"]))
        link_colors.append("rgba(170,170,170,0.35)")
        link_hover.append(f"theme {t}: {row['keywords']}<br>no sector ≥ {TAU_SHOW}")

fig_sankey = go.Figure(
    go.Sankey(
        arrangement="snap",
        node={
            "label": node_labels,
            "color": node_colors,
            "pad": 8,
            "thickness": 14,
            "line": {"color": "#333", "width": 0.4},
        },
        link={
            "source": sources,
            "target": targets,
            "value": values,
            "color": link_colors,
            "customdata": link_hover,
            "hovertemplate": "%{customdata}<extra></extra>",
        },
    )
)
fig_sankey.update_layout(
    title=f"Themes ↔ ICB Sectors (links shown when cos ≥ {TAU_SHOW}; right column stacked by Industry)",
    height=max(700, 22 * (n_themes + n_sectors)),
    font={"size": 11},
)

# --- Combined HTML: Sankey + per-theme drill-down ---
sim_to_centroid = news_emb @ centroids.T  # (N, n_topics)

parts = [
    "<html><head><meta charset='utf-8'><title>Themes v0</title>"
    "<style>"
    "body{font-family:-apple-system,Segoe UI,sans-serif;max-width:1200px;margin:24px auto;padding:0 16px;color:#222;}"
    "h1{margin-bottom:4px;}"
    "h2{margin-top:36px;border-bottom:1px solid #eee;padding-bottom:4px;}"
    "h3{margin-top:24px;}"
    ".meta{color:#666;font-size:13px;}"
    ".aligned{color:#1a7f37;font-weight:600;}"
    ".novel{color:#bf5700;font-weight:600;}"
    "li{margin:4px 0;}"
    "code{background:#f3f3f3;padding:1px 4px;border-radius:3px;}"
    "</style></head><body>",
    "<h1>Themes v0</h1>",
    f"<p class='meta'>{DATE_START[:10]} → {DATE_END[:10]} · "
    f"{len(norm_headlines):,} headlines · {len(themes)} themes · "
    f"embedder <code>{EMBEDDING_MODEL}</code> · τ_show={TAU_SHOW} · τ_cov={TAU_COV}</p>",
    fig_sankey.to_html(full_html=False, include_plotlyjs="cdn"),
    "<h2>Per-theme drill-down</h2>",
]

for _, row in themes.iterrows():
    t = int(row["topic"])
    col = valid_topics.index(t)
    in_topic = np.where(topics_arr == t)[0]
    scores = sim_to_centroid[in_topic, col]
    top_idx = in_topic[np.argsort(-scores)[:TOP_HEADLINES_PER_THEME]]
    cls = "aligned" if row["bucket"] == "aligned" else "novel"
    parts.append(
        f"<h3>Theme {t} — <span class='{cls}'>{row['bucket']}</span></h3>"
        f"<p class='meta'>{row['n_docs']:,} headlines · nearest sector "
        f"<b>{html_lib.escape(row['nearest_sector'])}</b> "
        f"({html_lib.escape(row['nearest_industry'])}) · cos={row['max_cos']:.2f}<br>"
        f"keywords: <code>{html_lib.escape(row['keywords'])}</code></p><ul>"
    )
    for i in top_idx:
        parts.append(f"<li>{html_lib.escape(norm_headlines[i])}</li>")
    parts.append("</ul>")
parts.append("</body></html>")

OUTPUT_HTML.write_text("\n".join(parts), encoding="utf-8")
print(f"Wrote {OUTPUT_HTML}")
fig_sankey.show()

Wrote /Users/federicocinus/Progetti - Local/ThematicTrading/repo/notebooks/output/themes_v0.html
